In [4]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

In [5]:
EU_COUNTRY_COORDS = {
    "Austria": (47.5162, 14.5501),
    "Belgium": (50.5039, 4.4699),
    "Bulgaria": (42.7339, 25.4858),
    "Croatia": (45.1000, 15.2000),
    "Cyprus": (35.1264, 33.4299),
    "Czech Republic": (49.8175, 15.4730),
    "Denmark": (56.2639, 9.5018),
    "Estonia": (58.5953, 25.0136),
    "Finland": (61.9241, 25.7482),
    "France": (46.2276, 2.2137),
    "Germany": (51.1657, 10.4515),
    "Greece": (39.0742, 21.8243),
    "Hungary": (47.1625, 19.5033),
    "Ireland": (53.7798, -7.3055),
    "Italy": (41.8719, 12.5674),
    "Latvia": (56.8796, 24.6032),
    "Lithuania": (55.1694, 23.8813),
    "Luxembourg": (49.8153, 6.1296),
    "Malta": (35.9375, 14.3754),
    "Netherlands": (52.1326, 5.2913),
    "Poland": (51.9194, 19.1451),
    "Portugal": (39.3999, -8.2245),
    "Romania": (45.9432, 24.9668),
    "Slovakia": (48.6690, 19.6990),
    "Slovenia": (46.1512, 14.9955),
    "Spain": (40.4637, -3.7492),
    "Sweden": (60.1282, 18.6435),
}

df = pd.DataFrame.from_dict(EU_COUNTRY_COORDS, orient="index", columns=["latitude", "longitude"])
df.index.name = "country"
eu_coords = pd.read_csv("../datafiles/eu_country_coords.csv")
print(df)

                latitude  longitude
country                            
Austria          47.5162    14.5501
Belgium          50.5039     4.4699
Bulgaria         42.7339    25.4858
Croatia          45.1000    15.2000
Cyprus           35.1264    33.4299
Czech Republic   49.8175    15.4730
Denmark          56.2639     9.5018
Estonia          58.5953    25.0136
Finland          61.9241    25.7482
France           46.2276     2.2137
Germany          51.1657    10.4515
Greece           39.0742    21.8243
Hungary          47.1625    19.5033
Ireland          53.7798    -7.3055
Italy            41.8719    12.5674
Latvia           56.8796    24.6032
Lithuania        55.1694    23.8813
Luxembourg       49.8153     6.1296
Malta            35.9375    14.3754
Netherlands      52.1326     5.2913
Poland           51.9194    19.1451
Portugal         39.3999    -8.2245
Romania          45.9432    24.9668
Slovakia         48.6690    19.6990
Slovenia         46.1512    14.9955
Spain            40.4637    

In [7]:
df = pd.read_csv("../datafiles/model_df.csv")

unique_cities = df["city"].dropna().unique()

geolocator = Nominatim(user_agent="uni_coords")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

coords = {}
for city in unique_cities:
    try:
        location = geocode(city)
        if location:
            coords[city] = (location.latitude, location.longitude)
        else:
            coords[city] = (None, None)
    except Exception as e:
        print(f"Failed for {city}: {e}")
        coords[city] = (None, None)

df["latitude"] = df["city"].map(lambda c: coords.get(c, (None, None))[0])
df["longitude"] = df["city"].map(lambda c: coords.get(c, (None, None))[1])

df.to_csv("../datafiles/model_df_with_coords.csv", index=False)
print(f"Done. {df['latitude'].notna().sum()} / {len(df)} universities have coordinates.")

Done. 702 / 780 universities have coordinates.
